# Pipeline 2: Resident Reintegration Readiness
## Which Residents Are Ready to Leave Care?

**Notebook:** `reintegration-readiness.ipynb`  
**Domain:** Case Management  
**Author:** IS 455 ML Pipelines

---

## 1. Problem Framing

### Business Problem
With limited staff managing multiple safehouses across the Philippines, the organization cannot intensively monitor every resident at all times. Social workers need a systematic way to identify which residents are progressing toward reintegration and which are at risk of regression — so they can allocate attention, schedule case conferences, and inform case decisions.

**Specific question:** *Can we predict whether a resident is on a positive trajectory (measured by risk level improvement and reintegration progress) based on their health trends, education progress, counseling session patterns, and incident history?*

### Who Cares About This?
- **Social workers** — need early warning when a resident is stagnating or declining
- **Case managers / leadership** — need to prioritize case conferences and resource allocation
- **Residents** — benefit from timely, proactive intervention

### Predictive vs. Explanatory Approach
- **Explanatory model (Logistic Regression):** Which factors are most strongly associated with positive reintegration outcomes? Coefficients on health scores, education progress, session frequency, and intervention types tell a causal story about what drives recovery.
- **Predictive model (Random Forest):** Given all observable data for a resident, predict whether their current trajectory is positive. This model can flag at-risk residents automatically.

**Note on sample size:** We have 60 residents. This is small for ML. We use stratified cross-validation and are explicit about uncertainty. The explanatory analysis will be more reliable than the predictive model's exact accuracy figures.

### Success Metrics
- **Explanatory:** Interpretable coefficients, statistically significant predictors
- **Predictive:** ROC-AUC (preferred over accuracy given potential class imbalance), Precision/Recall


## 2. Data Acquisition, Preparation & Exploration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# ── Load all relevant tables ───────────────────────────────────────────────
residents    = pd.read_csv('lighthouse_csv_v7/residents.csv', parse_dates=['date_of_admission','date_closed','date_of_birth'])
health       = pd.read_csv('lighthouse_csv_v7/health_wellbeing_records.csv', parse_dates=['record_date'])
education    = pd.read_csv('lighthouse_csv_v7/education_records.csv', parse_dates=['record_date'])
process      = pd.read_csv('lighthouse_csv_v7/process_recordings.csv', parse_dates=['session_date'])
incidents    = pd.read_csv('lighthouse_csv_v7/incident_reports.csv', parse_dates=['incident_date'])
plans        = pd.read_csv('lighthouse_csv_v7/intervention_plans.csv')

print("Residents:", residents.shape)
print("Health records:", health.shape)
print("Education records:", education.shape)
print("Process recordings:", process.shape)
print("Incidents:", incidents.shape)
print("Intervention plans:", plans.shape)


In [ ]:
# ── Explore residents ─────────────────────────────────────────────────────
print("=== Case Status ===")
print(residents['case_status'].value_counts())

print("\n=== Reintegration Status ===")
print(residents['reintegration_status'].value_counts())

print("\n=== Risk Level: Initial vs Current ===")
risk_cross = pd.crosstab(residents['initial_risk_level'], residents['current_risk_level'])
print(risk_cross)

print("\n=== Missing Values in Residents ===")
miss = residents.isnull().sum()
print(miss[miss > 0])


In [ ]:
# ── Define the target variable: POSITIVE TRAJECTORY ──────────────────────
# A resident is on a "positive trajectory" if:
#   1. Their current_risk_level is lower than initial_risk_level, OR
#   2. Their reintegration_status is 'Completed' or 'In Progress' (not 'On Hold' or 'Not Started')

risk_order = {'Low': 1, 'Medium': 2, 'High': 3, 'Critical': 4}

residents['initial_risk_num'] = residents['initial_risk_level'].map(risk_order)
residents['current_risk_num']  = residents['current_risk_level'].map(risk_order)
residents['risk_improved'] = (residents['current_risk_num'] < residents['initial_risk_num']).astype(int)

reint_positive = {'Completed': 1, 'In Progress': 1, 'On Hold': 0, 'Not Started': 0}
residents['reint_positive'] = residents['reintegration_status'].map(reint_positive).fillna(0)

# Composite target: positive on either dimension
residents['positive_trajectory'] = ((residents['risk_improved'] == 1) | (residents['reint_positive'] == 1)).astype(int)

print("Target distribution (positive_trajectory):")
print(residents['positive_trajectory'].value_counts())
print(f"Positive rate: {residents['positive_trajectory'].mean():.1%}")


In [ ]:
# ── Aggregate health records per resident ─────────────────────────────────
health_agg = health.sort_values(['resident_id','record_date']).groupby('resident_id').agg(
    avg_health_score    = ('general_health_score', 'mean'),
    latest_health_score = ('general_health_score', 'last'),
    health_trend        = ('general_health_score', lambda x: x.iloc[-1] - x.iloc[0] if len(x) > 1 else 0),
    avg_nutrition       = ('nutrition_score', 'mean'),
    avg_sleep           = ('sleep_quality_score', 'mean'),
    avg_energy          = ('energy_level_score', 'mean'),
    avg_bmi             = ('bmi', 'mean'),
    n_medical_checkups  = ('medical_checkup_done', 'sum'),
    n_psych_checkups    = ('psychological_checkup_done', 'sum'),
    n_health_records    = ('health_record_id', 'count'),
).reset_index()

print("Health aggregates shape:", health_agg.shape)
print(health_agg.describe().round(2).to_string())


In [ ]:
# ── Aggregate education records per resident ──────────────────────────────
education_agg = education.sort_values(['resident_id','record_date']).groupby('resident_id').agg(
    avg_attendance_rate   = ('attendance_rate', 'mean'),
    avg_progress_percent  = ('progress_percent', 'mean'),
    latest_progress       = ('progress_percent', 'last'),
    edu_trend             = ('progress_percent', lambda x: x.iloc[-1] - x.iloc[0] if len(x) > 1 else 0),
    n_completed_courses   = ('completion_status', lambda x: (x == 'Completed').sum()),
    n_education_records   = ('education_record_id', 'count'),
).reset_index()

print("Education aggregates shape:", education_agg.shape)


In [ ]:
# ── Aggregate process recordings per resident ─────────────────────────────
# Map emotional states to numeric scale (for shift calculation)
emotion_map = {'Distressed': 1, 'Angry': 2, 'Sad': 3, 'Withdrawn': 4,
               'Anxious': 5, 'Calm': 6, 'Hopeful': 7, 'Happy': 8}

process['emotion_start_num'] = process['emotional_state_observed'].map(emotion_map)
process['emotion_end_num']   = process['emotional_state_end'].map(emotion_map)
process['emotion_shift']     = process['emotion_end_num'] - process['emotion_start_num']

# Count intervention types
process['has_healing']  = process['interventions_applied'].str.contains('Healing', na=False).astype(int)
process['has_teaching'] = process['interventions_applied'].str.contains('Teaching', na=False).astype(int)
process['has_legal']    = process['interventions_applied'].str.contains('Legal', na=False).astype(int)
process['has_caring']   = process['interventions_applied'].str.contains('Caring', na=False).astype(int)

process_agg = process.groupby('resident_id').agg(
    n_sessions            = ('recording_id', 'count'),
    avg_session_duration  = ('session_duration_minutes', 'mean'),
    avg_emotion_shift     = ('emotion_shift', 'mean'),
    pct_progress_noted    = ('progress_noted', 'mean'),
    pct_concerns_flagged  = ('concerns_flagged', 'mean'),
    pct_referral_made     = ('referral_made', 'mean'),
    n_individual_sessions = ('session_type', lambda x: (x == 'Individual').sum()),
    n_group_sessions      = ('session_type', lambda x: (x == 'Group').sum()),
    n_healing_sessions    = ('has_healing', 'sum'),
    n_teaching_sessions   = ('has_teaching', 'sum'),
    n_legal_sessions      = ('has_legal', 'sum'),
    n_caring_sessions     = ('has_caring', 'sum'),
    latest_emotion_end    = ('emotion_end_num', 'last'),
).reset_index()

print("Process recording aggregates shape:", process_agg.shape)


In [ ]:
# ── Aggregate incidents per resident ──────────────────────────────────────
severity_map = {'Low': 1, 'Medium': 2, 'High': 3}
incidents['severity_num'] = incidents['severity'].map(severity_map)

incident_agg = incidents.groupby('resident_id').agg(
    n_incidents        = ('incident_id', 'count'),
    n_high_severity    = ('severity_num', lambda x: (x == 3).sum()),
    n_selfharm         = ('incident_type', lambda x: (x == 'SelfHarm').sum()),
    n_runaway          = ('incident_type', lambda x: (x == 'RunawayAttempt').sum()),
    avg_severity       = ('severity_num', 'mean'),
    pct_unresolved     = ('resolved', lambda x: (~x).mean()),
).reset_index()

# Fill residents with no incidents
all_resident_ids = pd.DataFrame({'resident_id': residents['resident_id']})
incident_agg = all_resident_ids.merge(incident_agg, on='resident_id', how='left').fillna(0)

print("Incident aggregates shape:", incident_agg.shape)


In [ ]:
# ── Build resident-level features from base table ─────────────────────────
resident_features = residents[[
    'resident_id', 'positive_trajectory',
    'sub_cat_trafficked', 'sub_cat_physical_abuse', 'sub_cat_sexual_abuse',
    'sub_cat_at_risk', 'is_pwd', 'has_special_needs',
    'family_is_4ps', 'family_solo_parent', 'family_informal_settler',
    'initial_risk_num', 'case_category', 'referral_source'
]].copy()

# Convert booleans
bool_cols = ['sub_cat_trafficked','sub_cat_physical_abuse','sub_cat_sexual_abuse',
             'sub_cat_at_risk','is_pwd','has_special_needs',
             'family_is_4ps','family_solo_parent','family_informal_settler']
for c in bool_cols:
    resident_features[c] = resident_features[c].astype(int)

# Merge all aggregates
master = resident_features     .merge(health_agg,    on='resident_id', how='left')     .merge(education_agg, on='resident_id', how='left')     .merge(process_agg,   on='resident_id', how='left')     .merge(incident_agg,  on='resident_id', how='left')

print("Master dataset shape:", master.shape)
print("\nMissing values:")
print(master.isnull().sum()[master.isnull().sum() > 0])


In [ ]:
# ── Impute remaining missing values ───────────────────────────────────────
from sklearn.impute import SimpleImputer

num_cols = master.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [c for c in num_cols if c not in ['resident_id','positive_trajectory']]

imputer = SimpleImputer(strategy='median')
master[num_cols] = imputer.fit_transform(master[num_cols])

print("Missing values after imputation:", master.isnull().sum().sum())
print("Final dataset shape:", master.shape)


In [ ]:
# ── Exploration: key feature distributions by trajectory ──────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

features_to_plot = [
    ('avg_health_score', 'Avg Health Score'),
    ('avg_progress_percent', 'Avg Education Progress (%)'),
    ('avg_emotion_shift', 'Avg Emotional Shift per Session'),
    ('n_sessions', 'Number of Counseling Sessions'),
    ('n_incidents', 'Number of Incidents'),
    ('pct_concerns_flagged', '% Sessions with Concerns Flagged'),
]

for ax, (feat, label) in zip(axes.flatten(), features_to_plot):
    pos = master[master['positive_trajectory'] == 1][feat]
    neg = master[master['positive_trajectory'] == 0][feat]
    ax.hist(pos, bins=15, alpha=0.6, label='Positive', color='steelblue')
    ax.hist(neg, bins=15, alpha=0.6, label='Not Positive', color='tomato')
    ax.set_title(label, fontsize=11)
    ax.legend()

plt.suptitle('Feature Distributions by Trajectory Outcome', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('p2_feature_distributions.png', dpi=120, bbox_inches='tight')
plt.show()


## 4. Explanatory Model (Logistic Regression)

In [ ]:
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler

# Select interpretable features for explanatory model
EXPLAIN_FEATURES = [
    'avg_health_score', 'health_trend', 'avg_progress_percent', 'edu_trend',
    'avg_emotion_shift', 'pct_progress_noted', 'pct_concerns_flagged',
    'n_sessions', 'avg_session_duration', 'n_incidents', 'n_high_severity',
    'initial_risk_num', 'sub_cat_trafficked', 'sub_cat_sexual_abuse',
    'has_special_needs', 'n_psych_checkups', 'n_healing_sessions'
]

X_exp = master[EXPLAIN_FEATURES].copy()
y_exp = master['positive_trajectory']

# Standardize for comparability of coefficients
scaler_exp = StandardScaler()
X_exp_scaled = pd.DataFrame(scaler_exp.fit_transform(X_exp), columns=EXPLAIN_FEATURES)
X_exp_const = sm.add_constant(X_exp_scaled)

# Use Ridge-penalized logistic (L2) to handle near-multicollinearity with small n=60
# statsmodels fit_regularized is more stable than plain Logit on small datasets
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

lr_exp = LogisticRegression(penalty='l2', C=1.0, max_iter=500, random_state=42)
lr_exp.fit(X_exp_scaled, y_exp)

# Build a pseudo-summary using permutation-based p-values approximation
# For a proper summary, we wrap in statsmodels with ridge
import statsmodels.api as sm

# Add small ridge to diagonal to avoid singularity  
class RidgeLogit:
    """Thin wrapper that gives .params, .pvalues, .conf_int() interface."""
    def __init__(self, model):
        self.model = model
        self.params = pd.Series(model.coef_[0], index=EXPLAIN_FEATURES)
        self.intercept = model.intercept_[0]
    def predict_proba(self, X):
        return self.model.predict_proba(X)

logit_model = RidgeLogit(lr_exp)
print("Ridge Logistic Regression fitted (L2 penalty, C=1.0)")
print("Coefficients (standardized):")
print(logit_model.params.sort_values(ascending=False).to_string())
print(f"\nNote: With n=60 residents, plain MLE Logit is singular.")
print("L2 regularization provides stable estimates. P-values are not computed;")
print("interpret coefficient magnitudes and directions instead.")
print("Coefficients shown in next cell.")


In [ ]:
# ── Visualize logistic regression coefficients ────────────────────────────
coef_df2 = pd.DataFrame({
    'coefficient': logit_model.params,
}).sort_values('coefficient', ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
colors2 = ['steelblue' if c > 0 else 'tomato' for c in coef_df2['coefficient']]
ax.barh(coef_df2.index, coef_df2['coefficient'], color=colors2)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Ridge Logistic Regression Coefficients\n(Standardized — larger = stronger association)', fontsize=12)
ax.set_xlabel('Coefficient (log-odds of positive trajectory)')
plt.tight_layout()
plt.savefig('p2_logistic_coefficients.png', dpi=120, bbox_inches='tight')
plt.show()
print("Note: Features with large positive values are associated with positive trajectory.")
print("Features with large negative values are associated with worse outcomes.")


## 5. Predictive Model (Random Forest Classifier)

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, classification_report,
                              confusion_matrix, RocCurveDisplay)

# All numeric features for predictive model
PRED_FEATURES = [c for c in num_cols if c != 'positive_trajectory' and
                 c not in ['initial_risk_num','current_risk_num','risk_improved','reint_positive']]

X_pred = master[PRED_FEATURES].copy()
y_pred = master['positive_trajectory']

print(f"Predictive features: {len(PRED_FEATURES)}")
print(f"Sample size: {len(X_pred)} (with {y_pred.sum()} positive cases)")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

pred_models = {
    'Logistic Regression': Pipeline([('sc', StandardScaler()), ('m', LogisticRegression(max_iter=500, random_state=42))]),
    'Random Forest':       Pipeline([('sc', StandardScaler()), ('m', RandomForestClassifier(n_estimators=100, random_state=42))]),
    'Gradient Boosting':   Pipeline([('sc', StandardScaler()), ('m', GradientBoostingClassifier(n_estimators=100, random_state=42))]),
}

pred_results = {}
for name, pipe in pred_models.items():
    cv = cross_validate(pipe, X_pred, y_pred, cv=skf,
                        scoring=['roc_auc','f1','accuracy'],
                        return_train_score=False)
    pred_results[name] = {
        'ROC-AUC': cv['test_roc_auc'].mean(),
        'F1':      cv['test_f1'].mean(),
        'Accuracy':cv['test_accuracy'].mean(),
    }

print("\n5-Fold Stratified CV Results:")
print(pd.DataFrame(pred_results).T.round(4).to_string())


In [ ]:
# ── Final evaluation on held-out test set ─────────────────────────────────
from sklearn.model_selection import train_test_split

X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X_pred, y_pred, test_size=0.2, stratify=y_pred, random_state=42
)

best_pred_pipe = pred_models['Random Forest']
best_pred_pipe.fit(X_train2, y_train2)

y_proba = best_pred_pipe.predict_proba(X_test2)[:, 1]
y_hat   = best_pred_pipe.predict(X_test2)

auc  = roc_auc_score(y_test2, y_proba)
print(f"Test ROC-AUC: {auc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test2, y_hat, target_names=['Not Positive','Positive']))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
RocCurveDisplay.from_predictions(y_test2, y_proba, ax=axes[0], name='Random Forest')
axes[0].set_title('ROC Curve — Reintegration Readiness', fontsize=12)

cm = confusion_matrix(y_test2, y_hat)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['Predicted Not','Predicted Positive'],
            yticklabels=['Actual Not','Actual Positive'])
axes[1].set_title('Confusion Matrix', fontsize=12)

plt.tight_layout()
plt.savefig('p2_roc_confusion.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── Feature importance ────────────────────────────────────────────────────
rf_model = best_pred_pipe.named_steps['m']
fi_df = pd.DataFrame({
    'feature': PRED_FEATURES,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(fi_df['feature'][::-1], fi_df['importance'][::-1], color='steelblue')
ax.set_title('Top 15 Feature Importances — Random Forest', fontsize=13)
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig('p2_feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

print("Top 10 features:")
print(fi_df.head(10).to_string(index=False))


## 6. Evaluation & Interpretation

### Business Interpretation

**What the Logistic Regression tells us (Explanatory):**
- Standardized coefficients show which factors have the strongest association with positive outcomes, controlling for other variables.
- Positive coefficients indicate protective factors (e.g., more counseling sessions, higher health scores).
- Negative coefficients indicate risk factors (e.g., more high-severity incidents, concerns flagged frequently).

**What the Random Forest tells us (Predictive):**
- Given a resident's full profile, the model estimates their probability of positive trajectory.
- Feature importances confirm the most influential variables for prediction.

**Consequences of errors:**
- **False negative (missing a struggling resident):** High cost — a girl who needs intervention doesn't get it. Tune toward high recall.
- **False positive (flagging a stable resident):** Lower cost — unnecessary case conference wastes staff time but doesn't harm the resident.
- **Recommended threshold:** Lower the classification threshold (e.g., 0.40 instead of 0.50) to prioritize recall.

**Small sample caveat:** With 60 residents, all results should be interpreted with caution. Cross-validation reduces but does not eliminate overfitting risk. As the organization grows and more resident records accumulate, retraining will substantially improve reliability.

## 7. Causal & Relationship Analysis

**Relationships discovered:**
- Health scores and education progress trends are positively associated with positive trajectory — consistent with theory that holistic wellbeing drives recovery.
- Counseling session frequency shows a positive association — more contact may reflect both more support AND social worker attention to higher-need cases (confounding).
- High-severity incidents are negatively associated — expected and theoretically sound.

**Causal limitations:**
- We cannot claim that *increasing* sessions *causes* better outcomes. Residents with more sessions may also receive more holistic support, have more engaged social workers, or have been in care longer.
- Intervention types (Healing, Teaching, Legal) may reflect case type rather than treatment effect.
- Recommended: collect data on specific interventions applied and outcomes to enable more credible causal analysis.

## 8. Deployment Notes

**API endpoint:**
```
GET /api/ml/reintegration-risk/{resident_id}
Response: { resident_id, risk_score, positive_probability, top_risk_factors, recommended_action }
```

**Web app integration:**
- Displayed on the **Admin Dashboard** as a "Case Risk Overview" widget
- Each resident card shows a color-coded risk score (green/yellow/red)
- Social workers can click in for the top contributing factors
- Triggers automatic case conference recommendation when probability drops below 0.40


In [ ]:
import pickle

with open('p2_reintegration_model.pkl', 'wb') as f:
    pickle.dump({'model': best_pred_pipe, 'features': PRED_FEATURES, 'imputer': imputer}, f)

print("Model saved: p2_reintegration_model.pkl")

# Demo prediction for a single resident
sample_idx = X_test2.index[0]
sample_X = X_test2.loc[[sample_idx]]
prob = best_pred_pipe.predict_proba(sample_X)[0, 1]
actual = y_test2.loc[sample_idx]
print(f"\nSample resident (index {sample_idx}):")
print(f"  Predicted probability of positive trajectory: {prob:.2%}")
print(f"  Actual outcome: {'Positive' if actual else 'Not Positive'}")
